# 02 — Load Delta → first Lakebase (as `app_admin`)
Resilient to Provisioned or Autoscale resource type. Runs in parallel with notebook 03.

In [0]:
# %pip install -q "psycopg[binary]>=3.1" "databricks-sdk>=0.81.0"
# dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("provisioned_instance_name", "lb-provisioned-demo")
dbutils.widgets.text("pg_database",               "databricks_postgres")
dbutils.widgets.text("target_table",              "public.events")
dbutils.widgets.text("delta_staging_table",       "main.default.lakebase_demo_events")
dbutils.widgets.text("admin_role",                "app_admin")
dbutils.widgets.text("secret_scope",              "lakebase_demo")
dbutils.widgets.text("secret_key",                "admin_password")

NAME     = dbutils.widgets.get("provisioned_instance_name")
PG_DB    = dbutils.widgets.get("pg_database")
TABLE    = dbutils.widgets.get("target_table")
DELTA    = dbutils.widgets.get("delta_staging_table")
USER     = dbutils.widgets.get("admin_role")
PWD      = dbutils.secrets.get(
              scope=dbutils.widgets.get("secret_scope"),
              key=dbutils.widgets.get("secret_key"),
           )

In [0]:
import uuid, psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound

w = WorkspaceClient()

def resolve_host(name: str) -> str:
    try:
        return w.database.get_database_instance(name=name).read_write_dns
    except NotFound:
        ep = w.postgres.get_endpoint(
            name=f"projects/{name}/branches/production/endpoints/ep-primary"
        )
        return ep.status.hosts.host

host = resolve_host(NAME)
dsn  = (f"host={host} port=5432 dbname={PG_DB} "
        f"user={USER} password={PWD} sslmode=require")
print(f"target host: {host}")

In [0]:
pdf = spark.table(DELTA).toPandas()
print(f"Loaded {len(pdf)} rows from {DELTA}")

rows = [
    (int(r.event_id), int(r.user_id), r.event_type, r.payload,
     r.event_ts.to_pydatetime())
    for r in pdf.itertuples(index=False)
]

INSERT_SQL = f"""
INSERT INTO {TABLE} (event_id, user_id, event_type, payload, event_ts)
VALUES (%s, %s, %s, %s::jsonb, %s)
ON CONFLICT (event_id) DO UPDATE SET
    user_id    = EXCLUDED.user_id,
    event_type = EXCLUDED.event_type,
    payload    = EXCLUDED.payload,
    event_ts   = EXCLUDED.event_ts;
"""

with psycopg.connect(dsn) as conn:
    with conn.cursor() as cur:
        cur.executemany(INSERT_SQL, rows)
    conn.commit()
    with conn.cursor() as cur:
        cur.execute(f"SELECT count(*) FROM {TABLE};")
        print(f"Row count on {NAME}: {cur.fetchone()[0]}")